# FinLLM GPU Training on Google Colab

This notebook runs the **exact same, already-tested code** from this repository - `models/`, `training/`, `data_sources/`, `evaluation/`, `ai_platform/model_registry.py` - on a real GPU. It does not reimplement any logic; every step below imports and calls the real project modules, the same way `main.py` does locally.

**Before running:** Runtime -> Change runtime type -> Hardware accelerator -> GPU.

**Honesty contract for this notebook:** every cell prints an actual measured value from the runtime. If a cell's assertion fails, STOP - do not skip ahead and do not let a later cell paper over an earlier failure. Section numbers below match the project's Colab GPU execution spec (Part 54).

## 1. Environment verification

In [ ]:
import sys, platform, os
print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', os.getcwd())

## 2-5. GPU, CUDA, PyTorch, and VRAM verification

**This is the hard gate.** If `torch.cuda.is_available()` is not `True`, STOP - do not proceed to claim GPU training happened.

In [ ]:
!nvidia-smi

In [ ]:
import torch

gpu_available = torch.cuda.is_available()
print('torch:', torch.__version__)
print('torch cuda build:', torch.version.cuda)
print('GPU AVAILABLE =', gpu_available)

if not gpu_available:
    raise RuntimeError(
        'STATUS = BLOCKED: torch.cuda.is_available() is False. '
        'Set Runtime -> Change runtime type -> GPU, then re-run from the top. '
        'Do not continue past this cell without a real GPU.'
    )

device_count = torch.cuda.device_count()
props = torch.cuda.get_device_properties(0)
vram_gb = round(props.total_memory / 1024**3, 2)
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
free_vram_gb = round(free_bytes / 1024**3, 2)

print('GPU name:', props.name)
print('device count:', device_count)
print('compute capability:', f'{props.major}.{props.minor}')
print('total VRAM GB:', vram_gb)
print('free VRAM GB (right now):', free_vram_gb)

GPU_NAME = props.name
VRAM_GB = vram_gb

## 6. Repository acquisition

Two real options - use whichever matches how you brought this notebook to Colab. **Note from this project's own history:** a git push from the machine that authored this notebook failed with a real 403 (wrong GitHub identity for this remote) - if `git clone` below fails the same way, that's a genuine access issue, not a notebook bug; use the archive-upload option instead.

In [ ]:
# OPTION A: clone from your own accessible GitHub remote (edit the URL)
# !git clone https://github.com/<your-username-or-fork>/DeepSeek-from-Scratch.git
# %cd DeepSeek-from-Scratch

# OPTION B: upload the repo as a zip via the Colab file browser (left sidebar), then:
# !unzip -q DeepSeek-from-Scratch.zip
# %cd DeepSeek-from-Scratch

print('Uncomment ONE option above and run it before continuing.')

In [ ]:
# Repository integrity check - confirm the critical paths actually
# came across, not just that *some* files did.
import os
required = ['models/model.py', 'models/config.py', 'training/trainer.py',
            'data_sources/dataset_registry.py', 'evaluation/evaluator.py',
            'ai_platform/model_registry.py', 'configs/model_config.yaml', 'main.py']
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise RuntimeError(f'Repository transfer incomplete - missing: {missing}')
print('Repository integrity: OK,', len(required), 'critical paths present')

## 7. Dependency installation

In [ ]:
# Colab ships a CUDA-enabled torch already - do NOT reinstall the CPU
# wheel from requirements.txt over it.
!pip install -q tiktoken datasets pyyaml pypdf python-docx scikit-learn networkx langdetect
!python -m pip check

In [ ]:
import torch, tiktoken, datasets, yaml, sklearn, networkx, langdetect
print('All required imports succeeded')
print('torch cuda still available after installs:', torch.cuda.is_available())

## 8-9. Dataset acquisition + validation

Uses the real, already-tested `data_sources.prepare_dataset` pipeline (streaming, bounded by `max_tokens`, clean -> dedup -> leakage-safe split -> shard) - not a simplified reimplementation for this notebook.

In [ ]:
from data_sources import prepare_dataset, list_entries

for e in list_entries(verified_only=True):
    print(e.name, '|', e.category, '|', e.license, '|', e.verification_status)

In [ ]:
# Larger budgets than were practical on CPU (small/financial_poc were
# explicitly never run locally - estimated ~300h/~2400h there).
summaries = {}
for name, tokens in [
    ('fineweb_edu', 10_000_000),
    ('financial_text_investopedia', 3_000_000),
    ('financial_qa_sujet', 3_000_000),
    ('financial_reasoning_finqa', 1_000_000),
    ('financial_reports_edgar', 3_000_000),
    ('financial_instruction_alpaca', 1_000_000),
]:
    summaries[name] = prepare_dataset(name=name, max_tokens=tokens)
    s = summaries[name]
    print(f"{name}: train={s['train_tokens_used']:,} val={s['validation_tokens_used']:,} "
          f"dup_removed={s['records_removed_duplicate']} invalid_removed={s['records_removed_invalid']}")

In [ ]:
# Dataset validation (Part 54.7): duplicates/invalid records are
# already reported per-source above (real numbers, not assumed zero).
# Now the cross-cutting check: eval data must NEVER be in the training
# shards - reuses the same leakage checker the CPU runs used.
from evaluation import check_leakage
leak = check_leakage()
print('Leakage check:', leak)
if not leak.get('clean', False):
    raise RuntimeError(f'STATUS = BLOCKED: evaluation data leaked into training shards: {leak}')
print('Dataset validation: CLEAN -', leak['checked_shards'], 'shard sets checked,',
      leak['eval_questions'], 'eval questions, 0 leaks')

## 10. Tokenizer setup

In [ ]:
from data_sources.tokenizer import get_encoding
enc = get_encoding()
probe = enc.encode_ordinary('What is EBITDA margin?')
print('vocab size:', enc.n_vocab)
print('probe encode:', probe)
print('probe decode:', repr(enc.decode(probe)))
assert enc.decode(probe) == 'What is EBITDA margin?'
print('Tokenizer round-trip: OK')

## 11. Model configuration

In [ ]:
from models import DeepSeekConfig, DeepSeekV3
config = DeepSeekConfig.default()
print(config)
model = DeepSeekV3(config).to('cuda')
total_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total_params:,}')
del model
torch.cuda.empty_cache()

## 12. Checkpoint discovery + validation

If you uploaded a prior checkpoint (e.g. the CPU-trained one from this project's earlier work) into `checkpoints/`, validate it for real before trusting it as a resume point - do not train from a checkpoint that merely *exists*.

In [ ]:
import glob, json
from ai_platform.model_registry import register_checkpoint, verify_integrity

found = glob.glob('checkpoints/*/*.pt')
print('checkpoints found:', found)

for path in found:
    try:
        ckpt = torch.load(path, map_location='cpu')
    except Exception as e:
        print(f'{path}: FAILED TO LOAD - {type(e).__name__}: {e}')
        continue
    state = ckpt.get('model_state_dict', ckpt)
    # Tensor-shape / architecture compatibility: every key in the
    # checkpoint must match a real parameter of a freshly-built model
    # with the CURRENT config, both by name and by shape.
    ref_model = DeepSeekV3(DeepSeekConfig.default())
    ref_params = dict(ref_model.named_parameters())
    shape_mismatches = []
    for k, v in state.items():
        if k not in ref_params:
            shape_mismatches.append(f'{k}: not in current architecture')
        elif tuple(v.shape) != tuple(ref_params[k].shape):
            shape_mismatches.append(f'{k}: checkpoint shape {tuple(v.shape)} != current {tuple(ref_params[k].shape)}')
    stage = path.split(os.sep)[-2]
    if shape_mismatches:
        print(f'{path}: INCOMPATIBLE with current architecture -', shape_mismatches[:3])
    else:
        entry = register_checkpoint(path, stage, set_active=False)
        v = verify_integrity(entry['version'])
        print(f"{path}: OK - step={ckpt.get('step')} params_match=True checksum_valid={v['valid']}")

## 13. Training configuration (auto-selected from real available VRAM)

Not a fixed guess - reads the VRAM this specific runtime actually reports (from section 2-5 above) and picks a batch size that should fit, erring conservative. If OOM still happens in the training cell, the notebook's OOM-recovery cell below reduces it further and retries - this cell is a starting point, not a guarantee.

In [ ]:
# Conservative heuristic: budget ~4GB of the reported VRAM for the
# model+optimizer+activations overhead, then size batch*seq from what's left.
# These are the same three presets already defined in configs/*.yaml -
# this cell only chooses which preset and what to override.
import yaml

if VRAM_GB >= 20:
    preset_name, batch_size, seq_len = 'financial_poc', 16, 1024
elif VRAM_GB >= 12:
    preset_name, batch_size, seq_len = 'small', 12, 512
elif VRAM_GB >= 6:
    preset_name, batch_size, seq_len = 'small', 6, 384
else:
    preset_name, batch_size, seq_len = 'tiny_debug', 4, 256

print(f'Selected preset={preset_name} batch_size={batch_size} seq_len={seq_len} '
      f'for {GPU_NAME} ({VRAM_GB} GB VRAM)')

# Apply the override on top of the checked-in preset file (in-memory only;
# does not modify the tracked configs/*.yaml).
with open(f'configs/{preset_name}.yaml') as f:
    preset = yaml.safe_load(f)
preset['batch_size'] = batch_size
preset['seq_len'] = seq_len
print(preset)

## 14-16. Training, validation, checkpoint saving

Runs the real `training.trainer.train_model` - the exact function `main.py train` calls locally. If CUDA OOM occurs, the recovery cell below captures it, halves batch size, and retries - it does not hide the error or fabricate a result.

In [ ]:
import yaml as _yaml
with open(f'configs/{preset_name}.yaml', 'w') as f:
    _yaml.dump(preset, f)

from training.trainer import train_model

def attempt_training(batch_size, seq_len, max_retries=3):
    for attempt in range(max_retries):
        try:
            with open(f'configs/{preset_name}.yaml') as f:
                p = _yaml.safe_load(f)
            p['batch_size'] = batch_size
            p['seq_len'] = seq_len
            with open(f'configs/{preset_name}.yaml', 'w') as f:
                _yaml.dump(p, f)
            print(f'Attempt {attempt+1}: batch_size={batch_size} seq_len={seq_len}')
            model, config, ckpt_path = train_model(preset_name=preset_name)
            return model, config, ckpt_path
        except torch.cuda.OutOfMemoryError as e:
            print(f'CUDA OOM at batch_size={batch_size}, seq_len={seq_len}: {e}')
            torch.cuda.empty_cache()
            if batch_size > 1:
                batch_size = max(1, batch_size // 2)
            elif seq_len > 128:
                seq_len = max(128, seq_len // 2)
            else:
                raise RuntimeError('STATUS = BLOCKED: OOM even at minimum batch_size=1, seq_len=128')
            print(f'Retrying with batch_size={batch_size} seq_len={seq_len}')
    raise RuntimeError('STATUS = BLOCKED: exhausted OOM retries')

model, config, ckpt_path = attempt_training(batch_size, seq_len)
print('Training completed. Checkpoint:', ckpt_path)

## Resume test (Part 54.11)

TRAIN -> SAVE -> STOP -> LOAD -> RESUME -> TRAIN MORE -> VERIFY. Actually executed, matching the same test already run on CPU earlier in this project (and which caught a real bug there: batch/seq sizes that OOM at train time OOM identically on resume).

In [ ]:
import json as _json
with open(ckpt_path.replace('.pt', '.json')) as f:
    meta_before = _json.load(f)
step_before = meta_before['step']
print('Checkpoint step before resume:', step_before)

model2, config2, ckpt_path2 = train_model(preset_name=preset_name, resume=ckpt_path)
with open(ckpt_path2.replace('.pt', '.json')) as f:
    meta_after = _json.load(f)
step_after = meta_after['step']
print('Checkpoint step after resume+more training:', step_after)

assert step_after > step_before, f'Resume did not advance: {step_before} -> {step_after}'
print(f'RESUME TEST: PASS ({step_before} -> {step_after})')

## 17. Evaluation

Runs the real Financial LLM POC Evaluation harness (`evaluation.evaluate_model`) - the same one whose scorer was independently validated (100% on oracle answers, 0% on wrong ones) earlier in this project.

In [ ]:
from evaluation import evaluate_model, print_report
results = evaluate_model(model2, device='cuda', max_new_tokens=48, verbose=True)
print_report(results)

## 18. Inference test - the exact required prompts

In [ ]:
from inference import load_model_for_inference, generate_text

loaded_model, loaded_config = load_model_for_inference(ckpt_path2, device='cuda')
print('Loaded independently from disk:', sum(p.numel() for p in loaded_model.parameters()), 'params')

prompts = [
    'What is EBITDA?',
    'Calculate EBITDA margin for revenue 500 and EBITDA 100.',
    'What is working capital?',
]
for p in prompts:
    out = generate_text(ckpt_path2, p, max_tokens=40, temperature=0.8, top_k=40, device='cuda')
    finite = all(c is not None for c in out)  # generation completed without a runtime error
    print(f'PROMPT: {p!r}')
    print(f'OUTPUT: {out!r}')
    print(f'completed without error: {finite}')
    print()

## 19. Checkpoint export

In [ ]:
from ai_platform.model_registry import register_checkpoint
import datetime

stage = 'base' if 'base' in ckpt_path2 else ('instruction' if 'instruction' in ckpt_path2 else 'financial')
entry = register_checkpoint(ckpt_path2, stage, set_active=False)

export_manifest = {
    'model_name': 'FinLLM-102M',
    'version': entry['version'],
    'checkpoint_path': ckpt_path2,
    'training_run_gpu': GPU_NAME,
    'training_run_vram_gb': VRAM_GB,
    'sha256': entry['checksum'],
    'size_bytes': entry['size_bytes'],
    'step': entry['step'],
    'train_loss': entry['train_loss'],
    'val_loss': entry['val_loss'],
    'evaluation_overall_accuracy': results['overall']['accuracy'],
    'exported_at': datetime.datetime.utcnow().isoformat() + 'Z',
}
with open('export_manifest.json', 'w') as f:
    _json.dump(export_manifest, f, indent=2)
print(_json.dumps(export_manifest, indent=2))

## 20. Artifact verification

Re-derive the checksum independently (not just trust the value computed during export) and confirm it matches.

In [ ]:
import hashlib

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(1024 * 1024)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

recomputed = sha256_file(ckpt_path2)
with open('export_manifest.json') as f:
    manifest = _json.load(f)

assert recomputed == manifest['sha256'], 'Artifact hash mismatch - export is not trustworthy'
print('Artifact verification: PASS - recomputed hash matches manifest')
print('sha256:', recomputed)

---
## Returning the checkpoint to the local project (Part 54.15-54.17)

1. Download `checkpoints/<stage>/checkpoint_<step>.pt` and its matching `.json`, plus `export_manifest.json`.
2. Place them into the local repo's `checkpoints/<stage>/` directory.
3. Run locally: `python -c "from ai_platform.model_registry import register_checkpoint; register_checkpoint('checkpoints/<stage>/checkpoint_<step>.pt', '<stage>')"` - this re-verifies the checksum on the LOCAL machine independently, and only then marks it as the active/serving version (never overwrite the existing production checkpoint before this passes).
4. Start the local backend (`python app/backend/server.py --checkpoint checkpoints/<stage>/checkpoint_<step>.pt`) and re-run the project's existing regression/acceptance tests (`ai_platform/acceptance_test.py`) against it, exactly as already proven against the CPU-trained checkpoint earlier in this project.
5. Only report `LLM TRAINING STATUS = TESTED` after that local regression pass succeeds - per Part 54.21, every item in that checklist must be true, not just the Colab-side steps.